#Tarea 2 - nGramas

In [ ]:
# Instalación y carga de archivos

# Objetivo de esta celda:
# Preparar el entorno para trabajar con:
# 1. spaCy analisis lingüístico (lematización y POS tagging)
# 2. NLTK n-gramas, colocaciones y PMI

import sys
import subprocess

try:
  import spacy
except ImportError:
  subprocess.check_call([sys.executable, "-m", "pip", "install", "spacy"])
  import spacy

# Cargar modelo en español
# spaCy necesita un modelo dl idioma a trabajar.
# "es_core__news_sm" es un modelo pequeño de español que ya sabe
# hacer tareas como tokenización, lematizacion y POS tagging.

try:
  nlp= spacy.load('es_core_news_sm')
except OSError:
  subprocess.check_call([sys.executable, "-m", "spacy", "download", "es_core_news_sm"])
  nlp= spacy.load('es_core_news_sm')

import nltk

from nltk.util import ngrams
from nltk.collocations import BigramCollocationFinder
from nltk.metrics import BigramAssocMeasures
from collections import Counter

In [ ]:
# Reseñas

dataset_vuelos_alumnos = [
    "El vuelo fue retrasado por 3 horas, pésimo servicio.",
    "La comida estaba fría y el asiento roto.",
    "El vuelo retrasado me hizo perder la conexión en Madrid.",
    "Excelente atención de la azafata, pero el asiento roto fue muy incómodo.",
    "Mi vuelo fue cancelado y el personal de tierra fue grosero.",
    "El vuelo retrasado es una falta de respeto al pasajero.",
    "Me cobraron doble por el equipaje extra. Pésimo servicio en mostrador.",
    "El asiento roto me lastimó la espalda, exijo reembolso del vuelo.",
    "Llamé a servicio al cliente por el equipaje extra y me colgaron.",
    "El vuelo cancelado arruinó mis vacaciones.",
    "Mi vuelo fue cancelado y nadie me dio solución.",
    "Pésimo servicio en mostrador y equipaje extra demasiado caro.",
    "El asiento roto fue muy incómodo durante todo el vuelo."
]

# ESCRIBE TU CÓDIGO AQUÍ ABAJO

In [ ]:
# 1. Procesa todas las reseñas usando limpiar_lematizar(...)

def limpiar_lematizar (texto):
  documento = nlp(texto.lower())
  tokens_limpios=[]

  etiquetas_validas={"NOUN", "ADJ", "VERB"}
  negaciones = {"no", "ni", "nunca"}

# Recorremos token por token dentro del documento
  for token in documento:
    # token.is_alpha == True si el token tiene letras.
    # is_alpha ignorara signos, puntuaciones y números
    if token.is_alpha:
      # Si es una negación imortate, la conservamos tal cual
      if token.text in negaciones:
        tokens_limpios.append(token.text)
      # Si el token pertenece a las categorias que si queremos, guardamos su lema
      elif token.pos_ in etiquetas_validas:
        tokens_limpios.append(token.lemma_)
  return tokens_limpios

In [ ]:
# 2. Junta todos los tokens en una sola lista

tokens_todos=[]

for resena in dataset_vuelos_alumnos:
  tokens_procesados = limpiar_lematizar(resena)

  tokens_todos.extend(tokens_procesados)

print("Tokens totales")
print(tokens_todos)

Tokens totales
['vuelo', 'retrasar', 'hora', 'pésimo', 'servicio', 'comida', 'frío', 'asiento', 'roto', 'vuelo', 'retrasado', 'hacer', 'perder', 'conexión', 'excelente', 'atención', 'azafata', 'asiento', 'roto', 'incómodo', 'vuelo', 'cancelar', 'personal', 'tierra', 'grosero', 'vuelo', 'retrasado', 'falta', 'respeto', 'pasajero', 'cobrar', 'doble', 'equipaje', 'extra', 'pésimo', 'servicio', 'mostrador', 'asiento', 'roto', 'lastimar', 'espalda', 'exijo', 'reembolso', 'vuelo', 'llamar', 'servicio', 'cliente', 'equipaje', 'extra', 'colgar', 'vuelo', 'cancelado', 'arruinar', 'vacación', 'vuelo', 'cancelar', 'dar', 'solución', 'pésimo', 'servicio', 'mostrador', 'equipaje', 'extra', 'caro', 'asiento', 'roto', 'incómodo', 'vuelo']


In [ ]:
# 3. Crea un BigramCollocationFinder con esa lista

buscador= BigramCollocationFinder.from_words(tokens_todos)

# 4. Aplica un filtro mínimo de frecuencia
buscador.apply_freq_filter(2)

metricas = BigramAssocMeasures()

# 5. Muestra:
#    - los 5 bigramas más frecuentes
print("\nTop 5 bigramas por frecuencia")
for bigrama, puntuacion in buscador.score_ngrams(metricas.raw_freq)[:5]:
  print(f"{bigrama}: {puntuacion:.4f}")

#    - los 5 bigramas con mayor PMI
print("\nTop 5 bigramas por PMI")
for bigrama, puntuacion in buscador.score_ngrams(metricas.pmi)[:5]:
  print(f"{bigrama}: {puntuacion:.4f}")


Top 5 bigramas por frecuencia
('asiento', 'roto'): 0.0588
('equipaje', 'extra'): 0.0441
('pésimo', 'servicio'): 0.0441
('incómodo', 'vuelo'): 0.0294
('roto', 'incómodo'): 0.0294

Top 5 bigramas por PMI
('equipaje', 'extra'): 4.5025
('asiento', 'roto'): 4.0875
('pésimo', 'servicio'): 4.0875
('roto', 'incómodo'): 4.0875
('servicio', 'mostrador'): 4.0875


In [ ]:
# 6. Responde:

#    a) ¿Qué problemas aparecen más por frecuencia?
#       El problema que aparece con mayor frecuencia es "asiento roto"",
#       con un puntaje de aproximadamente 0.0588.
#       Otros problemas frecuentes son "equipaje extra" y
#       "pésimo servicio", ambos con un puntaje de 0.0441.

#    b) ¿Qué conceptos específicos aparecen con PMI?
#       El primer concepto es "equipaje extra" con 4.5025 de vaoración,
#       en segundo lugar asiento roto" y en tercero "pésimo servicio"
#       empatados con aprox. 4.0875

#    c) ¿Por qué no siempre coinciden frecuencia y PMI?
#       Porque miden cosas distintas:
#       Frecuencia: cuenta cuántas veces aparecen las palabras juntas.
#       Si una palabra es muy común aparecerá en muchos bigramas solo por azar.
#       PMI Mide la fuerza de la asociación. Nos dice cuánto más probable es
#       encontrar las dos palabras juntas de lo que sería si fueran independientes.
#       Un bigrama como 'equipaje extra' puede tener un PMI alto porque,
#       aunque no ocurra tantas veces como 'asiento roto', cuando aparece
#       la palabra 'extra', casi siempre es para acompañar a 'equipaje'.